# Hello World — Claude Managed Agents (CLI)

The same minimal end-to-end flow as [`01-basics/`](../01-basics/), but driven entirely from the terminal with Anthropic's **`ant`** CLI instead of the Python SDK:

```
Environment (once) → Agent (once) → Session (every run) → Stream events
```

| Object | Lifecycle | Purpose |
|--------|-----------|----------|
| **Environment** | Create once | Sandboxed container where tools run (bash, files, code) |
| **Agent** | Create once | Versioned config: model, system prompt, tools |
| **Session** | Create per run | Links agent + environment; Anthropic runs the loop |

### CLI for the control plane, SDK for the data plane

The split this notebook teaches: **agents and environments are static resources** you define as version-controlled YAML and apply with `ant` (control plane). **Sessions are dynamic** — created per run and streamed (data plane). Both hit the same API; the difference is where the call lives.

> **In production:** persist `environment.id` and `agent.id` — don't recreate them on every run.

## 1. Setup

The CLI exposes every Claude API resource as a shell subcommand. Beta resources (agents, environments, sessions) live under the `beta:` prefix, and the CLI sets the right `anthropic-beta` header automatically.

First, make sure `ant` is installed.

In [ ]:
%%bash
# If `ant` isn't installed yet, install it (pick one):
#   macOS:        brew install anthropics/tap/ant
#   Linux / WSL:  download a release from github.com/anthropics/anthropic-cli/releases
#   From source:  go install github.com/anthropics/anthropic-cli/cmd/ant@latest
ant --version || echo "ant not found — install it using one of the methods above"


### Authentication

The CLI resolves credentials the same way the SDKs do (first match wins): `ANTHROPIC_API_KEY`, then `ANTHROPIC_AUTH_TOKEN`, then an `ant auth login` profile.

The easiest path in a notebook is to have **`ANTHROPIC_API_KEY` set in the environment** (the same key the Python notebooks read from the root `.env`). Alternatively, run `ant auth login` once to store an OAuth profile.

> **Trap:** profiles are only consulted when no API key is set. A stale exported `ANTHROPIC_API_KEY` silently overrides every profile. `ant auth status` shows which credential source won.

In [ ]:
%%bash
# Shows which credential source and workspace the CLI will use (status only).
ant auth status || echo "Set ANTHROPIC_API_KEY or run: ant auth login"


## 2. Create an Environment

The environment is the **sandboxed container** where tools execute. We define it as version-controlled YAML — the recommended control-plane flow — then apply it with `ant`.

- `type: cloud` — Anthropic manages the infra
- `networking.type: unrestricted` — allows outbound internet access

**Create this once and reuse the ID.**

In [ ]:
%%writefile env.yaml
name: basics-env-cli
config:
  type: cloud
  networking:
    type: unrestricted


Apply it. `--transform id -r` extracts just the `id` field as a bare string (no quotes), which we capture into a Python variable so later cells can reuse it — the CLI analog of storing `environment.id`.

In [ ]:
# Pipe the YAML in via stdin; capture the new environment's id.
_env = !ant beta:environments create --transform id -r < env.yaml
env_id = _env[0]
print("Environment ID:", env_id)


## 3. Create an Agent

The agent is a **persisted, versioned config**: model, system prompt, and tools. `agent_toolset_20260401` is Anthropic's prebuilt toolset (bash, file ops, code execution, web search/fetch) — all running inside the environment container.

Every update creates a new **immutable version**, so sessions can pin to a specific version and never break.

In [ ]:
%%writefile agent.yaml
name: Hello World Agent (CLI)
model: claude-opus-4-7
system: |
  You are a helpful assistant. Keep your answers concise.
tools:
  - type: agent_toolset_20260401
    default_config:
      enabled: true


We capture the full JSON response here so we can grab **both** the `id` and the `version` — sessions pin to an explicit version.

In [ ]:
import json

# --format json prints the full object; capture id + version for explicit pinning.
_agent = !ant beta:agents create --format json < agent.yaml
agent = json.loads("".join(_agent))
agent_id, agent_version = agent["id"], agent["version"]
print("Agent ID :", agent_id)
print("Version  :", agent_version)


## 4. Create a Session

A session is a **single run** — it links the agent to the environment. This is the data plane: created per run, streamed, and driven by events.

We pin to the exact `agent.version` so future agent updates don't affect this session, and stash the session id in an environment variable (`SID`) so the streaming `bash` cell below can read it.

In [ ]:
import os, json

# Pin the agent to its exact version: {"type": "agent", "id": ..., "version": ...}
agent_ref = json.dumps({"type": "agent", "id": agent_id, "version": agent_version})

_session = !ant beta:sessions create --agent '{agent_ref}' --environment-id {env_id} --title "Hello World CLI session" --transform id -r
session_id = _session[0]

os.environ["SID"] = session_id  # exported to the bash cell below
print("Session ID:", session_id)


## 5. Stream Events

This is the **stream-first pattern** — the headline rule of Managed Agents:

1. **Open the stream first** (`ant beta:sessions:events stream`) — on a file descriptor, before sending anything
2. **Then send the user message** (`ant beta:sessions:events send`) — triggers the agent loop
3. **Read events as they arrive** — handle `agent.message` text until the session goes idle

If you send before opening the stream, you'll miss early events.

| Event type | Meaning |
|---|---|
| `agent.message` | Claude's response text |
| `session.status_idle` | Agent finished its turn |
| `session.status_terminated` | Session is done |

The `--transform` flag (a [GJSON path](https://github.com/tidwall/gjson/blob/master/SYNTAX.md)) keeps only the fields we care about per event, and `--format yaml` makes them easy to read line by line — no `jq` needed.

In [ ]:
%%bash
set -uo pipefail

# 1. Open the event stream FIRST (stream-before-send), on file descriptor $stream.
exec {stream}< <(ant beta:sessions:events stream --session-id "$SID" \
  --transform '{type, text: content.#(type=="text").text, err: error.message}' \
  --format yaml)

# 2. Now send the user message — this triggers the agent loop.
ant beta:sessions:events send --session-id "$SID" >/dev/null <<'YAML'
events:
  - type: user.message
    content:
      - type: text
        text: Say hello and tell me one fun fact about octopuses.
YAML

# 3. Read events as they stream in; break when the session goes idle or ends.
printf 'Agent: '
type=
while IFS= read -r -u "$stream" line; do
  case "$line" in
    "type: session.status_idle")       break ;;   # agent finished its turn
    "type: session.status_terminated") break ;;   # session ended
    "type: session.error")
      IFS= read -r -u "$stream" next || next=
      case "$next" in "err: "*) msg=${next#err: } ;; *) msg=unknown ;; esac
      printf '\n[Error: %s]\n' "$msg"; break ;;
    "type: "*) type=${line#type: } ;;
    "text: "*)
      [ "$type" = agent.message ] || continue
      val=${line#text: }
      # YAML may emit multi-line text as a "|-" block scalar; the body arrives
      # on the indented continuation lines handled below.
      case "$val" in "|-"|"|") ;; *) printf '%s' "$val" ;; esac ;;
    "  "*)
      [ "$type" = agent.message ] && printf '%s\n' "${line#  }" ;;
  esac
done
exec {stream}<&-   # close the stream
printf '\n'


## Summary

You just ran the full Claude Managed Agents flow from the terminal:

```
ant beta:environments create < env.yaml   →  store the environment id
ant beta:agents create < agent.yaml        →  store agent id + version
ant beta:sessions create --agent ...        →  new session per run
ant beta:sessions:events stream             →  open stream FIRST
ant beta:sessions:events send               →  send message, triggers loop
read loop                                   →  handle agent.message until idle
```

This mirrors [`01-basics/`](../01-basics/) line for line — the Python SDK version does the same thing in code. Use whichever fits: the CLI for ad-hoc control-plane work and version-controlled YAML in CI, the SDK for application code that drives sessions.

### CLI cheatsheet

```bash
ant beta:agents list --transform '{id,name,model}' --format jsonl   # inspect agents
ant beta:sessions:events list --session-id "$SID" --transform 'content.0.text' -r
ant beta:agents update --agent-id "$AGENT_ID" --version N < agent.yaml   # new version
ant --help                                                          # explore
```

### Next steps
- **02-multi-turn** — keep the session alive and send follow-up messages
- **03-tools** — use bash and file tools inside the container
- **04-mcp** — connect external MCP servers to the agent